<a href="https://colab.research.google.com/github/Johnogunlola/MRes-AI/blob/MRes/Pax_load_factor_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:


import os, re, math, json, warnings, itertools
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pandas.tseries.offsets import MonthEnd


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.inspection import permutation_importance

In [ ]:
import statsmodels.api as sm
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [ ]:
# XGBoost
try:
    import xgboost as xgb
    XGB_OK = True
except Exception:
    XGB_OK = False


In [ ]:
# LSTM (TensorFlow/Keras)
try:
    import tensorflow as tf
    from tensorflow.keras import layers, callbacks, models
    TF_OK = True
except Exception:
    TF_OK = False


In [ ]:
# Network (for degree centrality visuals)
try:
    import networkx as nx
    NX_OK = True
except Exception:
    NX_OK = False


In [ ]:
PAX_FILE = "Monthly Domestic Air Pax Route Analysis by Each Reporting Airport 2015 - 2025.csv"
AIRPORTS_FILE = "uk_airports_icao_only.csv"
LF_FILE = "Quarterly Passenger Flights Load Factor Data Other Reporting Apts 2019 Q1.xlsx"



In [ ]:
OUT = Path("outputs")
VIZ = OUT / "visuals"
MODELS = OUT / "models"
for d in [OUT, VIZ, MODELS]:
    d.mkdir(parents=True, exist_ok=True)


In [ ]:
SEED = 42
np.random.seed(SEED)

def normalize_text(s: str) -> str:
    if pd.isna(s): return ""
    s = s.upper().strip()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"\(.*?\)", "", s).strip()
    s = s.replace(" AIRPORT", "")
    s = s.replace(" INTERNATIONAL", "")
    s = s.replace(" LONDON AREA AIRPORTS", "LONDON")
    return s


In [ ]:
def month_to_dt(yyyymm: str) -> pd.Timestamp:
    return pd.to_datetime(str(yyyymm) + "01") + MonthEnd(1)


In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0088
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = phi2 - phi1
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))


In [ ]:
def seasonal_naive_forecast(series: pd.Series, horizon: int, season: int = 12):
    if len(series) < season:
        vals = [series.iloc[-1]] * horizon
    else:
        last_season = series.iloc[-season:]
        vals = np.resize(last_season.values, horizon)
    idx = pd.date_range(series.index[-1] + MonthEnd(1), periods=horizon, freq="M")
    return pd.Series(vals, index=idx)


In [ ]:
def metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = (np.abs((y_true - y_pred) / np.maximum(y_true, 1e-9))).mean() * 100
    return {"MAE": mae, "RMSE": rmse, "MAPE%": mape}

In [ ]:
def add_time_feats(df, date_col="date"):
    out = df.copy()
    out["year"] = out[date_col].dt.year
    out["month"] = out[date_col].dt.month
    out["quarter"] = out[date_col].dt.quarter
    out["mon_sin"] = np.sin(2*np.pi*out["month"]/12)
    out["mon_cos"] = np.cos(2*np.pi*out["month"]/12)
    return out


In [ ]:
def make_lags(df, col="pax", lags=(1,3,6,12)):
    out = df.copy().sort_values("date")
    for L in lags:
        out[f"{col}_lag{L}"] = out[col].shift(L)
    return out


In [ ]:
def scale01(x):
    x = x.astype(float)
    return (x - np.nanmin(x)) / (np.nanmax(x) - np.nanmin(x) + 1e-9)


In [ ]:
pax = pd.read_csv(PAX_FILE)
airports = pd.read_csv(AIRPORTS_FILE)
lf_raw = pd.read_excel(LF_FILE, engine="openpyxl", header=None) # Read without header

# Manually construct column names from the second and third rows of the raw data (index 1 and 2)
new_columns = []
for i in range(lf_raw.shape[1]): # Iterate through all columns
    col_name_row1 = str(lf_raw.iloc[1, i]).strip() # Header candidate from row 1 (e.g., Year, Quarter)
    col_name_row2 = str(lf_raw.iloc[2, i]).strip() # Header candidate from row 2 (e.g., bucket names)

    if col_name_row1 != 'nan' and col_name_row1 != '':
        # For columns like 'Year', 'Quarter', 'Reporting Airport', 'Direction', 'Total'
        new_columns.append(col_name_row1)
    elif col_name_row2 != 'nan' and col_name_row2 != '':
        # For specific load factor bucket names
        new_columns.append(col_name_row2)
    else:
        new_columns.append(f'Unnamed_{i}') # Fallback

lf_raw.columns = new_columns
lf_raw = lf_raw[3:].reset_index(drop=True) # Drop the first three rows (header rows and any blank/descriptive rows)

In [ ]:
keep_cols = [
    "this_period","group_name","airport_1_name","airport_2_name",
    "total_pax_this_period","total_pax_scheduled_this_period","total_pax_charter_this_period"
]
pax = pax[keep_cols].copy()


In [ ]:
for c in ["group_name","airport_1_name","airport_2_name"]:
    pax[c] = pax[c].astype(str).apply(normalize_text)


In [ ]:
pax.dropna(subset=["this_period"], inplace=True)
pax["date"] = pax["this_period"].astype(int).astype(str).apply(month_to_dt)
pax = pax.rename(columns={
    "airport_1_name":"origin",
    "airport_2_name":"dest",
    "total_pax_this_period":"pax",
    "total_pax_scheduled_this_period":"pax_scheduled",
    "total_pax_charter_this_period":"pax_charter"
})

In [ ]:
pax = pax[pax["pax"].fillna(0) > 0]
pax["route_ud"] = pax.apply(lambda r: " — ".join(sorted([r["origin"], r["dest"]])), axis=1)


In [ ]:
pax["covid_shock"] = ((pax["date"] >= "2020-03-31") & (pax["date"] <= "2020-12-31")).astype(int)
pax["covid_recovery"] = ((pax["date"] >= "2021-01-31") & (pax["date"] <= "2022-12-31")).astype(int)


In [ ]:
panel_dir = (pax.groupby(["date","origin","dest","route_ud"], as_index=False)
             .agg(pax=("pax","sum"),
                  pax_scheduled=("pax_scheduled","sum"),
                  pax_charter=("pax_charter","sum")))


In [ ]:
panel_ud = (panel_dir.groupby(["date","route_ud"], as_index=False)
            .agg(pax=("pax","sum")))


In [ ]:
airports["name_n"] = airports["name"].astype(str).apply(normalize_text)
airports["municipality_n"] = airports["municipality"].astype(str).apply(normalize_text)
airports["is_island_hint"] = airports["name"].str.contains(
    r"ISLE|ISLAND|ISLES|STORNOWAY|KIRKWALL|SUMBURGH|BENBECULA|ISLAY|TIREE|JERSEY|GUERNSEY|ALDERNEY|BARRA",
    case=False, regex=True
)


In [ ]:

def match_airport(name_n: str):
    # exact
    hit = airports.loc[airports["name_n"]==name_n]
    if len(hit)==1: return hit.iloc[0]
    # contains on name
    hit = airports.loc[airports["name_n"].str.contains(fr"\b{name_n}\b", regex=True, na=False)]
    if len(hit)==1: return hit.iloc[0]
    # municipality exact
    hit = airports.loc[airports["municipality_n"]==name_n]
    if len(hit)==1: return hit.iloc[0]
    # contains on municipality
    hit = airports.loc[airports["municipality_n"].str.contains(fr"\b{name_n}\b", regex=True, na=False)]
    if len(hit)>0: return hit.iloc[0]
    return None

In [ ]:
unique_airports = pd.Index(pd.unique(pd.concat([panel_dir["origin"], panel_dir["dest"]]))).tolist()
rows = []
for nm in unique_airports:
    rec = match_airport(nm)
    if rec is None:
        rows.append({"name_n": nm, "lat": np.nan, "lon": np.nan, "icao": None, "iata":
None, "is_island_hint": False, "type": None})
    else:
        rows.append({
            "name_n": nm,
            "lat": rec.get("latitude_deg", np.nan),
            "lon": rec.get("longitude_deg", np.nan),
            "icao": rec.get("icao", None),
            "iata": rec.get("iata_code", None),
            "is_island_hint": bool(rec.get("is_island_hint", False)),
            "type": rec.get("type", None),
            "scheduled_service": rec.get("scheduled_service", None)
        })
airport_map = pd.DataFrame(rows)


In [ ]:
panel_dir = (panel_dir.merge(airport_map.add_prefix("o_"), left_on="origin", right_on="o_name_n", how="left")
                      .merge(airport_map.add_prefix("d_"), left_on="dest",   right_on="d_name_n", how="left"))

panel_dir["distance_km"] = haversine_km(panel_dir["o_lat"], panel_dir["o_lon"], panel_dir["d_lat"], panel_dir["d_lon"])


In [ ]:
sns.set_style("whitegrid")

In [ ]:
nat = pax.groupby("date", as_index=False)["pax"].sum()
plt.figure(figsize=(12,4))
plt.plot(nat["date"], nat["pax"], color="#1f77b4")
plt.axvspan(pd.Timestamp("2020-03-31"), pd.Timestamp("2020-12-31"), color="red", alpha=0.12, label="COVID shock")
plt.title("UK Domestic Passenger Demand (Monthly)")
plt.ylabel("Passengers"); plt.legend(); plt.tight_layout()
plt.savefig(VIZ/"01_national_monthly.png", dpi=180)


In [ ]:
nat["year"] = nat["date"].dt.year; nat["month"] = nat["date"].dt.month
yoy = (nat.pivot_table(index="month", columns="year", values="pax")
          .pct_change(axis=1)*100)
plt.figure(figsize=(12,4))
yoy.plot(kind="line", cmap="coolwarm", alpha=0.8) # Changed 'area' to 'line'
plt.title("YoY % Growth by Month"); plt.ylabel("%"); plt.tight_layout()
plt.savefig(VIZ/"02_yoy_growth_area.png", dpi=180); plt.close()

In [ ]:
season_idx = nat.assign(m=nat["date"].dt.month).groupby("m")["pax"].mean()/nat["pax"].mean()
plt.figure(figsize=(8,4)); season_idx.plot(kind="bar", color="#2f6fab")
plt.title("Seasonality Index (Avg=1.0)"); plt.tight_layout()
plt.savefig(VIZ/"03_seasonality_index.png", dpi=180); plt.close()


In [ ]:
nat_ts = nat.set_index("date")["pax"].asfreq("M").interpolate()
decomp = sm.tsa.seasonal_decompose(nat_ts, model="multiplicative", period=12)
fig = decomp.plot(); fig.set_size_inches(12,8)
plt.tight_layout(); plt.savefig(VIZ/"04_seasonal_decomposition.png", dpi=180); plt.close()


In [ ]:
top_routes = panel_ud.groupby("route_ud")["pax"].mean().sort_values(ascending=False).head(15).index
plt.figure(figsize=(11,6))
sns.lineplot(data=panel_ud[panel_ud["route_ud"].isin(top_routes)], x="date", y="pax", hue="route_ud", legend=False)
plt.title("Top 15 Routes – Monthly Passengers"); plt.tight_layout()
plt.savefig(VIZ/"05_top15_routes_ts.png", dpi=180); plt.close()


In [ ]:
top_nodes = pd.concat([panel_dir["origin"], panel_dir["dest"]]).value_counts().head(25).index.tolist()
mat = (panel_dir[panel_dir["origin"].isin(top_nodes) & panel_dir["dest"].isin(top_nodes)]
       .groupby(["origin","dest"])["pax"].sum().unstack(fill_value=0))
plt.figure(figsize=(12,10))
sns.heatmap(np.log1p(mat), cmap="viridis")
plt.title("Route Flow Heatmap (log pax) – Top 25 Airports"); plt.tight_layout()
plt.savefig(VIZ/"06_route_flow_heatmap.png", dpi=180); plt.close()


In [ ]:
mat_m = nat.pivot_table(index=nat["date"].dt.year, columns=nat["date"].dt.month, values="pax")
plt.figure(figsize=(10,6)); sns.heatmap(np.log1p(mat_m), cmap="magma")
plt.title("National Monthly Pax Heatmap (log)"); plt.tight_layout()
plt.savefig(VIZ/"07_national_monthly_heatmap.png", dpi=180); plt.close()


In [ ]:
route_meta = (panel_dir.groupby("route_ud", as_index=False)
              .agg(distance_km=("distance_km","median"),
                   avg_pax=("pax","mean")))
plt.figure(figsize=(7,5))
sns.scatterplot(data=route_meta, x="distance_km", y="avg_pax", alpha=0.5)
plt.title("Distance vs Avg Monthly Pax"); plt.tight_layout()
plt.savefig(VIZ/"08_dist_vs_avgpax.png", dpi=180); plt.close()


In [ ]:
tmp = panel_ud.copy(); tmp["m"] = tmp["date"].dt.month
plt.figure(figsize=(10,4)); sns.boxplot(data=tmp, x="m", y="pax")
plt.title("Distribution of Route Pax by Month"); plt.tight_layout()
plt.savefig(VIZ/"09_box_by_month.png", dpi=180); plt.close()


In [ ]:
ex_route = top_routes[0]
s = panel_ud[panel_ud["route_ud"]==ex_route].set_index("date")["pax"].asfreq("M").fillna(0)
rmean, rstd = s.rolling(12).mean(), s.rolling(12).std()
plt.figure(figsize=(10,4)); plt.plot(s.index, s, alpha=0.5, label="pax")
plt.plot(rmean.index, rmean, label="12m mean"); plt.plot(rstd.index, rstd, label="12m std")
plt.title(f"Rolling Stats – {ex_route}"); plt.legend(); plt.tight_layout()
plt.savefig(VIZ/"10_rolling_stats_example_route.png", dpi=180); plt.close()


In [ ]:
if NX_OK:
    G = nx.DiGraph()
    for _, r in panel_dir.groupby(["origin","dest"])["pax"].sum().reset_index().iterrows():
        G.add_edge(r["origin"], r["dest"], weight=r["pax"])
    deg = [d for n,d in G.degree()]
    plt.figure(figsize=(7,4)); plt.hist(deg, bins=20, color="#7aa974")
    plt.title("Airport Degree Distribution"); plt.tight_layout()
    plt.savefig(VIZ/"11_degree_distribution.png", dpi=180); plt.close()


In [ ]:
tmp2 = panel_dir.copy()
tmp2["month"] = tmp2["date"].dt.month
corr = tmp2[["pax","pax_scheduled","pax_charter","distance_km","month"]].corr()
plt.figure(figsize=(6,5)); sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix (sample features)"); plt.tight_layout()
plt.savefig(VIZ/"12_corr_matrix.png", dpi=180); plt.close()


def pre_post(df):
    pre = df[df["date"]<="2020-02-29"]["pax"].mean()
    post= df[df["date"]>="2021-01-31"]["pax"].mean()
    return (post - pre) / (pre+1e-9) * 100
pp = (panel_ud.groupby("route_ud").apply(pre_post).sort_values(ascending=True).head(20)).reset_index(name="pct_change")
plt.figure(figsize=(7,6)); sns.barplot(data=pp, x="pct_change", y="route_ud", palette="coolwarm")
plt.title("Biggest negative post-vs-pre changes (%) – Top 20"); plt.tight_layout()
plt.savefig(VIZ/"13_post_vs_pre_losses.png", dpi=180); plt.close()


In [ ]:
pg = (panel_ud.groupby("route_ud").apply(pre_post).sort_values(ascending=False).head(20)).reset_index(name="pct_change")
plt.figure(figsize=(7,6)); sns.barplot(data=pg, x="pct_change", y="route_ud", palette="RdYlGn")
plt.title("Biggest positive post-vs-pre changes (%) – Top 20"); plt.tight_layout()
plt.savefig(VIZ/"14_post_vs_pre_gains.png", dpi=180); plt.close()


In [ ]:
top_air = pd.Series(pd.concat([panel_dir["origin"], panel_dir["dest"]])).value_counts().head(10).index
thr = (panel_dir[panel_dir["origin"].isin(top_air)]
       .groupby(["date","origin"])["pax"].sum().reset_index())
plt.figure(figsize=(11,6))
sns.lineplot(data=thr, x="date", y="pax", hue="origin")
plt.title("Throughput by Airport (Top 10)"); plt.tight_layout()
plt.savefig(VIZ/"15_airport_throughput.png", dpi=180); plt.close()


In [ ]:
share = (thr.pivot_table(index="date", columns="origin", values="pax", aggfunc="sum").fillna(0))
share = share.div(share.sum(axis=1), axis=0)
share.plot.area(figsize=(12,6), cmap="tab20")
plt.title("Monthly Share of Pax – Top 10 Airports"); plt.tight_layout()
plt.savefig(VIZ/"16_share_top_airports.png", dpi=180); plt.close()


In [ ]:
plt.figure(figsize=(7,4)); plt.hist(route_meta["distance_km"].dropna(), bins=30, color="#6c7a89")
plt.title("Route Distance Distribution (km)"); plt.tight_layout()
plt.savefig(VIZ/"17_distance_hist.png", dpi=180); plt.close()


In [ ]:
plt.figure(figsize=(7,4)); plt.hist(np.log1p(panel_ud["pax"]), bins=40, color="#9467bd")
plt.title("Log Pax Distribution across Route-Months"); plt.tight_layout()
plt.savefig(VIZ/"18_log_pax_hist.png", dpi=180); plt.close()


In [ ]:
exm = panel_ud[panel_ud["route_ud"]==ex_route].copy()
exm["y"] = exm["date"].dt.year; exm["m"] = exm["date"].dt.month
cal = exm.pivot_table(index="y", columns="m", values="pax")
plt.figure(figsize=(8,5)); sns.heatmap(np.log1p(cal), cmap="YlGnBu")
plt.title(f"Calendar Heatmap (log pax) – {ex_route}"); plt.tight_layout()
plt.savefig(VIZ/"19_calendar_heatmap_example.png", dpi=180); plt.close()


In [ ]:
pm = panel_ud.merge(route_meta, on="route_ud", how="left")
pm["band"] = pd.cut(pm["distance_km"], [0,250,500,1000,5000], labels=["<250","250-500","500-1000","1000+"])
pm["m"] = pm["date"].dt.month
prof = pm.groupby(["band","m"])["pax"].mean().reset_index()
g = sns.relplot(data=prof, x="m", y="pax", hue="band", kind="line", height=4, aspect=1.8)
g.figure.suptitle("Seasonal Profiles by Distance Band", y=1.02)
g.savefig(VIZ/"20_season_by_distance.png", dpi=180)


In [ ]:
# ========= 6. FEATURE TABLE FOR MODELLING =========
# Merge meta into undirected table
route_meta2 = (panel_dir.groupby("route_ud", as_index=False)
               .agg(distance_km=("distance_km","median"),
                    island_any=("o_is_island_hint","max")))
mdf = panel_ud.merge(route_meta2, on="route_ud", how="left")
mdf = add_time_feats(mdf, "date")
mdf["covid_shock"] = ((mdf["date"] >= "2020-03-31") & (mdf["date"] <= "2020-12-31")).astype(int)
mdf["covid_recovery"] = ((mdf["date"] >= "2021-01-31") & (mdf["date"] <= "2022-12-31")).astype(int)
mdf = mdf.groupby("route_ud", group_keys=False).apply(lambda d: make_lags(d, "pax"))


In [ ]:
# ========= 7. TRAIN/TEST SPLIT & MODEL ZOO (BASELINE, SARIMAX, XGB, LSTM) =========
HOLDOUT = 12              # months for out-of-sample
MIN_LEN = 60              # minimum history to model
MAX_ROUTES_LSTM = 30      # to keep runtime reasonable
HORIZON = 12              # forecast horizon for deployment

results = []
per_route_preds = []
feature_set = ["pax_lag1","pax_lag3","pax_lag6","pax_lag12","mon_sin","mon_cos","distance_km","covid_shock","covid_recovery"]


In [ ]:
def eval_models_for_route(df_r, rid):
    df_r = df_r.sort_values("date").dropna(subset=[c for c in df_r.columns if "lag" in c])
    if len(df_r) < (MIN_LEN + HOLDOUT):
        return None
    train, test = df_r.iloc[:-HOLDOUT], df_r.iloc[-HOLDOUT:]
    Xtr, ytr = train[feature_set], train["pax"]
    Xte, yte = test[feature_set], test["pax"]

    # Baseline: Seasonal-Naive
    s = df_r.set_index("date")["pax"]
    sn = seasonal_naive_forecast(s.iloc[:-HOLDOUT], horizon=HOLDOUT, season=12)
    sn_pred = pd.Series(sn.values, index=test["date"])
    m_sn = metrics(yte.values, sn_pred.values)

    # SARIMAX (fast grid)
    best_sar, best_rmse, sar_pred = None, np.inf, None
    for order in [(1,1,1),(2,1,1),(1,1,2)]:
        try:
            sar = SARIMAX(ytr, order=order, seasonal_order=(1,1,1,12), enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
            pred = sar.get_forecast(steps=HOLDOUT).predicted_mean
            rmse = mean_squared_error(yte.values, pred.values, squared=False)
            if rmse < best_rmse:
                best_rmse, best_sar, sar_pred = rmse, sar, pred
        except Exception:
            pass

    if sar_pred is None:
        sar_pred = pd.Series([ytr.iloc[-1]]*HOLDOUT, index=test.index)
    else:
        sar_pred = pd.Series(sar_pred.values, index=test.index)
    m_sar = metrics(yte.values, sar_pred.values)

    # XGBoost / GradientBoosting
    if XGB_OK:
        xg = xgb.XGBRegressor(
            n_estimators=600, max_depth=4, learning_rate=0.04,
            subsample=0.9, colsample_bytree=0.9, objective="reg:squarederror", random_state=SEED
        )

        xg.fit(Xtr, ytr)
        xg_pred = xg.predict(Xte)
        m_xg = metrics(yte.values, xg_pred)
        xg_model = xg
    else:
        gbr = GradientBoostingRegressor(random_state=SEED)
        gbr.fit(Xtr, ytr)
        xg_pred = gbr.predict(Xte)
        m_xg = metrics(yte.values, xg_pred)
        xg_model = gbr

    # LSTM (optional & limited number of routes)
    m_lstm = {"MAE": np.nan, "RMSE": np.nan, "MAPE%": np.nan}
    lstm_pred = None
    if TF_OK:
        # Build windowed supervised data from y (univariate) to forecast next step recursively
        # For fair comparison we train on train set and predict full HOLDOUT recursively
        y_series = df_r["pax"].values.astype(np.float32)
        scaler = StandardScaler()
        ys = scaler.fit_transform(y_series.reshape(-1,1)).ravel()

        def make_windows(y, win=12):

            Xw, yw = [], []
            for i in range(win, len(y)):
                Xw.append(y[i-win:i])
                yw.append(y[i])
            return np.array(Xw), np.array(yw)

        WIN = 12
        Xw, yw = make_windows(ys[:-HOLDOUT], WIN)
        Xw = Xw[..., None]  # [samples, steps, 1]

        model = models.Sequential([
            layers.Input(shape=(WIN,1)),
            layers.LSTM(64, return_sequences=True),
            layers.LSTM(32),
            layers.Dense(1)
        ])
        model.compile(optimizer="adam", loss="mae")
        es = callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
        model.fit(Xw, yw, epochs=60, batch_size=16, validation_split=0.2, callbacks=[es], verbose=0)

        # Recursive forecast for HOLDOUT
        hist = ys[:-HOLDOUT].tolist()
        preds = []
        for _ in range(HOLDOUT):
            x_in = np.array(hist[-WIN:]).reshape(1,WIN,1)

            yhat = model.predict(x_in, verbose=0).ravel()[0]
            hist.append(yhat)
            preds.append(yhat)
        lstm_pred = scaler.inverse_transform(np.array(preds).reshape(-1,1)).ravel()
        m_lstm = metrics(yte.values, lstm_pred)

    # Pick best model by RMSE
    comp = [("SeasonalNaive", m_sn, sn_pred.values),
            ("SARIMAX", m_sar, sar_pred.values),
            ("XGB/GBR", m_xg, xg_pred if xg_pred is not None else np.full_like(yte.values, ytr.iloc[-1]))]
    if TF_OK and lstm_pred is not None:
        comp.append(("LSTM", m_lstm, lstm_pred))
    best = sorted(comp, key=lambda z: z[1]["RMSE"] if not np.isnan(z[1]["RMSE"]) else np.inf)[0]

    # Save a diagnostic plot
    plt.figure(figsize=(10,4))
    plt.plot(train["date"], ytr.values, label="Train", color="#8ba1b7")
    plt.plot(test["date"], yte.values, label="Actual", color="#1f77b4")
    plt.plot(test["date"], best[2], label=f"Pred ({best[0]})", color="#d62728")
    plt.title(f"{rid} – Holdout Forecast – Best: {best[0]}")
    plt.legend(); plt.tight_layout()
    fpath = VIZ / f"fc_{re.sub(r'[^A-Z0-9]+','_',rid)}.png"
    plt.savefig(fpath, dpi=170); plt.close()

    # Save metrics row
    out = {
        "route_ud": rid,
        "best_model": best[0],
        "sn_RMSE": m_sn["RMSE"], "sn_MAPE%": m_sn["MAPE%"],
        "sar_RMSE": m_sar["RMSE"], "sar_MAPE%": m_sar["MAPE%"],
        "xg_RMSE": m_xg["RMSE"], "xg_MAPE%": m_xg["MAPE%"],
        "lstm_RMSE": m_lstm["RMSE"], "lstm_MAPE%": m_lstm["MAPE%"] if TF_OK else np.nan,
    }
    return out



In [ ]:
long_routes = [rid for rid, g in mdf.groupby("route_ud") if len(g.dropna(subset=["pax_lag12"])) >= (MIN_LEN + HOLDOUT)]

In [ ]:
# Sort by historical total pax (desc) so LSTM will cover the most important ones first if needed
route_totals = panel_ud.groupby("route_ud")["pax"].sum().sort_values(ascending=False).index.tolist()
targets = [r for r in route_totals if r in long_routes]

lstm_counter = 0
for rid in targets:
    # temporarily toggle TF_OK if exceeding cap
    _TF = TF_OK and (lstm_counter < MAX_ROUTES_LSTM)
    if _TF: lstm_counter += 1
    # temporarily shadow global TF_OK in function? Use closure trick:
    TF_OK = _TF
    res = eval_models_for_route(mdf[mdf["route_ud"]==rid], rid)

    # restore TF_OK
    TF_OK = True if "tensorflow" in globals() else False
    if res: results.append(res)

results_df = pd.DataFrame(results).sort_values("sar_RMSE")
results_df.to_csv(OUT/"model_accuracy_comparison_by_route.csv", index=False)

# 21) Feature importance (XGB/GBR) on a representative route
if XGB_OK or len(results_df):
    sample_route = results_df["route_ud"].iloc[0] if len(results_df) else ex_route
    df_r = mdf[mdf["route_ud"]==sample_route].sort_values("date").dropna(subset=["pax_lag12"])
    tr = df_r.iloc[:-HOLDOUT]; Xtr, ytr = tr[feature_set], tr["pax"]
    if XGB_OK:

        model = xgb.XGBRegressor(n_estimators=600, max_depth=4, learning_rate=0.05,
                                 subsample=0.9, colsample_bytree=0.9, random_state=SEED, objective="reg:squarederror")
        model.fit(Xtr, ytr)
        imp = model.feature_importances_
        s = pd.Series(imp, index=feature_set).sort_values(ascending=True)
    else:
        model = GradientBoostingRegressor(random_state=SEED).fit(Xtr, ytr)
        pi = permutation_importance(model, Xtr, ytr, n_repeats=10, random_state=SEED)
        s = pd.Series(pi.importances_mean, index=feature_set).sort_values(ascending=True)
    plt.figure(figsize=(6,4)); s.plot(kind="barh", color="#2ca02c")

    plt.title(f"Feature Importance – {sample_route}")
    plt.tight_layout(); plt.savefig(VIZ/"21_feature_importance.png", dpi=180);
plt.close()





In [ ]:
# ========= 8. DEPLOYMENT FORECASTS (NEXT 12M) =========
def forecast_next_12m(df_r, rid):
    df_r = df_r.sort_values("date")
    last = df_r["date"].max()
    fut_idx = pd.date_range(last + MonthEnd(1), periods=HORIZON, freq="M")
    # Seasonal naive as baseline
    s = df_r.set_index("date")["pax"]
    base = seasonal_naive_forecast(s, horizon=HORIZON, season=12)

    # Build a quick SARIMAX for deployed forecast (robust, light)
    try:
        sar = SARIMAX(s, order=(1,1,1), seasonal_order=(1,1,1,12), enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
        sar_fc = sar.get_forecast(steps=HORIZON).predicted_mean
        sar_fc.index = fut_idx
        fc = sar_fc
    except Exception:
        fc = base

    return pd.DataFrame({"date": fut_idx, "route_ud": rid, "fc_pax": fc.values, "baseline_pax": base.values})

futures = []
for rid, grp in mdf.groupby("route_ud"):
    futures.append(forecast_next_12m(grp, rid))
future_fc = pd.concat(futures, ignore_index=True)
future_fc.to_csv(OUT/"future_12m_forecasts_by_route.csv", index=False)





In [ ]:
# ---------- Helper functions for robust LF Excel parsing ----------

def _normalize_col_label(c: str) -> str:
    """
    Normalise a raw column label to a comparable string:
    - remove newlines
    - collapse spaces
    - strip
    - lower case
    """
    c = str(c).replace("\n", " ")
    c = re.sub(r"\s+", " ", c).strip().lower()
    return c

def _canonicalize_lf_columns(raw_cols):
    """
    Map many real-world variants to canonical LF column names used downstream.
    Returns: dict {raw_col_name -> canonical_name}
    """
    aliases = {
        r"^year$": "Year",
        r"^quarter$": "Quarter",
        r"^reporting.*airport$": "Reporting Airport",
        r"^direction$": "Direction",
        r"^origin.*destination.*country$": "Origin/Destination Airport Country",
        r"^total$|^number of flights$": "Total",

        # LF buckets (percentage of flights in each bucket)
        r"^empty.*zero passengers.*cargo.*$|^empty.*cargo.*$|^empty.*$":
            "Empty (Zero Passengers and Cargo on Board)",
        r"^zero passengers.*cargo.*$":
            "Zero Passengers but Cargo on Board",
        r"^more than.*zero.*less than\s*10$|^>0.*<\s*10$|^0\-10%?$":
            "More than zero and Less than 10",
        r"^more.*equal\s*10.*less than\s*50$|^>=\s*10.*<\s*50$|^10\-50%?$":
            "More than or Equal 10 and Less than 50",
        r"^more.*equal\s*50$|^>=\s*50%?$|^50%\+?$":
            "More Than or Equal 50",
    }

    canon = {}
    for raw in raw_cols:
        key = _normalize_col_label(raw)
        mapped = None
        for pat, tgt in aliases.items():
            if re.search(pat, key):
                mapped = tgt
                break
        if mapped is None:
            # keep original if no match (we won't rely on it for melt)
            mapped = str(raw)
        canon[raw] = mapped
    return canon

def _read_lf_excel_robust(xlsx_path: str, sheet_name=0, search_rows=30):
    """
    Read the LF Excel even when headers are not on the first row or contain odd formatting.
    - Scans the first `search_rows` rows to detect the header row
    - Renames columns to canonical names via regex aliases
    """
    # Read everything with no header first
    df0 = pd.read_excel(xlsx_path, engine="openpyxl", sheet_name=sheet_name, header=None)

    # Expected core header tokens to look for in a row
    header_tokens = ["year", "quarter", "reporting", "direction"]
    header_idx = None
    for i in range(min(search_rows, len(df0))):
        row_texts = df0.iloc[i, :].astype(str).tolist()
        norm = [_normalize_col_label(x) for x in row_texts]
        score = sum(any(tok in x for x in norm) for tok in header_tokens)
        if score >= 3:  # row likely contains header fields
            header_idx = i
            break

    if header_idx is None:
        # Fallback: assume first non-empty row
        header_idx = 0

    # Re-read with detected header row
    lf = pd.read_excel(xlsx_path, engine="openpyxl", sheet_name=sheet_name, header=header_idx)

    # Canonicalize column names
    colmap = _canonicalize_lf_columns(lf.columns)
    lf = lf.rename(columns=colmap)

    # Persist the discovered (canonical) columns for debugging
    (OUT / "debug").mkdir(exist_ok=True, parents=True)
    pd.Series(lf.columns).to_csv(OUT / "debug" / "lf_columns_after_canonicalization.csv", index=False, header=False)

    return lf

In [ ]:
# ========= 9. LOAD FACTOR (LF) ESTIMATION & PREDICTION  (ROBUST) =========

# 9.1 Read & normalise the LF workbook robustly
lf = _read_lf_excel_robust(LF_FILE)

# Keep domestic rows (works with canonical name created above)
country_col = "Origin/Destination Airport Country"
if country_col not in lf.columns:
    raise ValueError(
        f"Expected column '{country_col}' not found after normalisation. "
        f"See outputs/debug/lf_columns_after_canonicalization.csv for actual columns."
    )
lf = lf[lf[country_col].astype(str).str.upper().str.contains("UNITED KINGDOM", na=False)].copy()

# Define the canonical LF bucket labels we use downstream
bucket_cols_canonical = [
    "Empty (Zero Passengers and Cargo on Board)",
    "Zero Passengers but Cargo on Board",
    "More than zero and Less than 10",
    "More than or Equal 10 and Less than 50",
    "More Than or Equal 50"
]

# Some files may not contain all buckets; keep only those that exist
present_buckets = [c for c in bucket_cols_canonical if c in lf.columns]
if len(present_buckets) == 0:
    raise ValueError("No LF bucket columns were detected in the Excel after normalisation.")

# Build id_vars dynamically from the canonical names that exist
id_candidates = ["Year", "Quarter", "Reporting Airport", "Direction", "Total", country_col]
id_vars = [c for c in id_candidates if c in lf.columns]

# Melt to long
lf_long = lf.melt(id_vars=id_vars,
                  value_vars=present_buckets,
                  var_name="bucket",
                  value_name="pct")

# Clean numeric
lf_long["pct"] = pd.to_numeric(lf_long["pct"], errors="coerce")

# Bucket mid-points for weighted LF estimate
bucket_mid = {
    "Empty (Zero Passengers and Cargo on Board)": 0.00,
    "Zero Passengers but Cargo on Board": 0.00,
    "More than zero and Less than 10": 0.05,
    "More than or Equal 10 and Less than 50": 0.30,
    "More Than or Equal 50": 0.75,
}

# Weighted average LF per reporting airport (ignore Direction for a single airport-level proxy)
if "Reporting Airport" not in lf_long.columns:
    # If the file lacks this column (unlikely), bail gracefully with a fallback
    # Use a single national average across all rows.
    national_lf = np.nansum((lf_long["pct"].values/100.0) * lf_long["bucket"].map(bucket_mid).values)
    lf_avg = pd.DataFrame({"Reporting Airport": ["__NATIONAL__"], "avg_lf_2019q1": [national_lf]})
else:
    lf_long["mid"] = lf_long["bucket"].map(bucket_mid)
    lf_avg = (lf_long.groupby(["Reporting Airport"])
              .apply(lambda d: np.nansum((d["pct"].values/100.0) * d["mid"].values))
              .reset_index(name="avg_lf_2019q1"))

# Build a normalised name key
lf_avg["Reporting Airport n"] = lf_avg["Reporting Airport"].astype(str).apply(normalize_text)
lf_map = lf_avg.set_index("Reporting Airport n")["avg_lf_2019q1"].to_dict()

# Fallback national LF if airport not found
FALLBACK_LF = float(np.nanmean(list(lf_map.values()))) if len(lf_map)>0 else 0.78

def est_route_lf(o, d):
    o_lf = lf_map.get(normalize_text(o), np.nan)
    d_lf = lf_map.get(normalize_text(d), np.nan)
    lf_est = np.nanmean([o_lf, d_lf])
    if np.isnan(lf_est): lf_est = FALLBACK_LF
    return float(np.clip(lf_est, 0.35, 0.95))

# Apply LF proxy to the historical route-direction rows
panel_dir["lf_proxy_base"] = panel_dir.apply(lambda r: est_route_lf(r["origin"], r["dest"]), axis=1)

# 9.2 Time-adjust LF using airport throughput vs 2019 baseline
ap_month = (panel_dir.groupby(["date","origin"])["pax"].sum().reset_index()
            .rename(columns={"origin":"airport","pax":"ap_pax"}))
ap_month["year"] = ap_month["date"].dt.year
ap_month["month"] = ap_month["date"].dt.month
base2019 = (ap_month[ap_month["year"]==2019]
            .groupby(["airport","month"])["ap_pax"].mean().reset_index()
            .rename(columns={"ap_pax":"ap_pax_2019m"}))
ap_month = ap_month.merge(base2019, on=["airport","month"], how="left")
ap_month["ap_scale"] = ap_month["ap_pax"] / (ap_month["ap_pax_2019m"] + 1e-9)
ap_scale_map = ap_month.set_index(["date","airport"])["ap_scale"].to_dict()

def lf_time_adjust(row):
    o_s = ap_scale_map.get((row["date"], row["origin"]), np.nan)
    d_s = ap_scale_map.get((row["date"], row["dest"]), np.nan)
    s = np.nanmean([o_s, d_s])
    if np.isnan(s): s = 1.0
    return float(np.clip(row["lf_proxy_base"] * s, 0.25, 0.97))

panel_dir["lf_proxy"] = panel_dir.apply(lf_time_adjust, axis=1)
panel_dir["seats_est"] = panel_dir["pax"] / np.maximum(panel_dir["lf_proxy"], 1e-6)

# 9.3 ML prediction of LF (per-route) + LF forecast (unchanged downstream)
lf_df = panel_dir[["date","route_ud","origin","dest","pax","distance_km","lf_proxy"]].copy()
lf_df = add_time_feats(lf_df, "date")
lf_df["covid_shock"] = ((lf_df["date"] >= "2020-03-31") & (lf_df["date"] <= "2020-12-31")).astype(int)
lf_df["covid_recovery"] = ((lf_df["date"] >= "2021-01-31") & (lf_df["date"] <= "2022-12-31")).astype(int)

lf_feats = ["pax","distance_km","mon_sin","mon_cos","covid_shock","covid_recovery"]
lf_models, lf_metrics = {}, []
for rid, g in lf_df.groupby("route_ud"):
    g = g.sort_values("date")
    if len(g) < 36: continue
    tr, te = g.iloc[:-HOLDOUT], g.iloc[-HOLDOUT:]
    Xtr, ytr = tr[lf_feats], tr["lf_proxy"]
    Xte, yte = te[lf_feats], te["lf_proxy"]

    if XGB_OK:
        model = xgb.XGBRegressor(
            n_estimators=400, max_depth=3, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9, random_state=SEED,
            objective="reg:squarederror"
        )
    else:
        model = GradientBoostingRegressor(random_state=SEED)

    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    me = metrics(yte.values, pred)
    me["route_ud"] = rid; lf_metrics.append(me)
    lf_models[rid] = model

lf_metrics_df = pd.DataFrame(lf_metrics)
lf_metrics_df.to_csv(OUT/"lf_model_metrics.csv", index=False)

# (Optional: small visual stays the same)
plt.figure(figsize=(7,4))
plt.hist(panel_dir["lf_proxy"].clip(0,1), bins=30, color="#ff7f0e")
plt.title("Route-Month LF Proxy Distribution"); plt.tight_layout()
plt.savefig(VIZ/"22_lf_proxy_distribution.png", dpi=180); plt.close()

In [ ]:
# 9.2 Adjust LF over time using a scaling factor (airport domestic throughput vs 2019 baseline)
# Build airport-month totals
ap_month = (panel_dir.groupby(["date","origin"])["pax"].sum().reset_index()
            .rename(columns={"origin":"airport","pax":"ap_pax"}))
ap_month["year"] = ap_month["date"].dt.year; ap_month["month"] = ap_month["date"].dt.month
base2019 = ap_month[ap_month["year"]==2019].groupby(["airport","month"])["ap_pax"].mean().reset_index().rename(columns={"ap_pax":"ap_pax_2019m"})
ap_month = ap_month.merge(base2019, on=["airport","month"], how="left")
ap_month["ap_scale"] = ap_month["ap_pax"] / (ap_month["ap_pax_2019m"]+1e-9)
ap_scale_map = ap_month.set_index(["date","airport"])["ap_scale"].to_dict()

def lf_time_adjust(row):
    o_s = ap_scale_map.get((row["date"], row["origin"]), np.nan)
    d_s = ap_scale_map.get((row["date"], row["dest"]), np.nan)
    s = np.nanmean([o_s, d_s])

    return float(np.clip(row["lf_proxy_base"] * s, 0.25, 0.97))

panel_dir["lf_proxy"] = panel_dir.apply(lf_time_adjust, axis=1)
panel_dir["seats_est"] = panel_dir["pax"] / np.maximum(panel_dir["lf_proxy"], 1e-6)



In [ ]:
# 9.3 Predict LF with ML (learn mapping from features → lf_proxy) & forecast LF
lf_df = panel_dir[["date","route_ud","origin","dest","pax","distance_km","lf_proxy"]].copy()
lf_df = add_time_feats(lf_df, "date")
lf_df["covid_shock"] = ((lf_df["date"] >= "2020-03-31") & (lf_df["date"] <= "2020-12-31")).astype(int)
lf_df["covid_recovery"] = ((lf_df["date"] >= "2021-01-31") & (lf_df["date"] <= "2022-12-31")).astype(int)

lf_feats = ["pax","distance_km","mon_sin","mon_cos","covid_shock","covid_recovery"]
lf_models = {}
lf_metrics = []
for rid, g in lf_df.groupby("route_ud"):
    g = g.sort_values("date")
    if len(g) < 36: continue
    tr, te = g.iloc[:-HOLDOUT], g.iloc[-HOLDOUT:]
    Xtr, ytr = tr[lf_feats], tr["lf_proxy"]
    Xte, yte = te[lf_feats], te["lf_proxy"]

    if XGB_OK:
        model = xgb.XGBRegressor(
            n_estimators=400, max_depth=3, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9, random_state=SEED, objective="reg:squarederror"
        )
    else:
        model = GradientBoostingRegressor(random_state=SEED)

    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    me = metrics(yte.values, pred)
    me["route_ud"] = rid; lf_metrics.append(me)
    lf_models[rid] = model

lf_metrics_df = pd.DataFrame(lf_metrics)

lf_metrics_df.to_csv(OUT/"lf_model_metrics.csv", index=False)

# 22) LF proxy distribution & predicted vs actual on sample
plt.figure(figsize=(7,4))
plt.hist(panel_dir["lf_proxy"].clip(0,1), bins=30, color="#ff7f0e")
plt.title("Route-Month LF Proxy Distribution"); plt.tight_layout()
plt.savefig(VIZ/"22_lf_proxy_distribution.png", dpi=180); plt.close()

if len(lf_metrics_df):
    rid = lf_metrics_df.sort_values("RMSE").iloc[0]["route_ud"]
    g = lf_df[lf_df["route_ud"]==rid].sort_values("date")
    tr, te = g.iloc[:-HOLDOUT], g.iloc[-HOLDOUT:]
    model = lf_models[rid]
    pred = model.predict(te[lf_feats])
    plt.figure(figsize=(10,4))
    plt.plot(te["date"], te["lf_proxy"], label="Actual LF", color="#1f77b4")
    plt.plot(te["date"], pred, label="Pred LF", color="#d62728")

    plt.title(f"LF Prediction – {rid}")
    plt.legend(); plt.tight_layout()
    plt.savefig(VIZ/"23_lf_pred_vs_actual.png", dpi=180); plt.close()





In [ ]:
# ========= 10. CAPACITY ACTIONS (DEMAND + LF) =========
# Combine next-12M demand forecasts with LF predictions
# For simplicity, use lf_model (if available) on future pax + time feats to project LF; otherwise keep last median LF.
future_fc["m"] = pd.to_datetime(future_fc["date"]).dt.month
future_fc["year"] = pd.to_datetime(future_fc["date"]).dt.year
# Merge distance and last known LF
lf_med = panel_dir.groupby("route_ud")["lf_proxy"].median().reset_index(name="lf_proxy_med")
future_fc = future_fc.merge(route_meta2[["route_ud","distance_km"]], on="route_ud", how="left")
future_fc = future_fc.merge(lf_med, on="route_ud", how="left")
# Predict LF if model available
def predict_future_lf(r):
    rid = r["route_ud"]

    if rid in lf_models:
        mon_sin = math.sin(2*math.pi*r["m"]/12.0); mon_cos = math.cos(2*math.pi*r["m"]/12.0)
        X = pd.DataFrame([{
            "pax": r["fc_pax"],
            "distance_km": r["distance_km"],
            "mon_sin": mon_sin,
            "mon_cos": mon_cos,
            "covid_shock": 0,
            "covid_recovery": 0
        }])
        return float(lf_models[rid].predict(X)[0])
    return float(r["lf_proxy_med"])

future_fc["lf_pred"] = future_fc.apply(predict_future_lf, axis=1)
future_fc["seats_needed_est"] = future_fc["fc_pax"] / np.maximum(future_fc["lf_pred"], 1e-6)

# Capacity rule
rule = []
for rid, g in future_fc.groupby("route_ud"):
    next12 = g["fc_pax"].sum()
    last12 = (panel_ud[panel_ud["route_ud"]==rid].tail(12)["pax"].sum())
    ratio = next12/(last12+1e-9)
    lf_m = g["lf_pred"].median()
    if (ratio >= 1.15) and (lf_m >= 0.85):
        action = "Increase frequency / upgauge"
    elif (ratio <= 0.85) or (lf_m <= 0.60):
        action = "Reduce frequency / downgauge"
    else:
        action = "Maintain / Monitor"
    rule.append({"route_ud": rid, "next12_vs_last12_ratio": ratio, "lf_median_pred": lf_m, "action": action})
capacity_actions = pd.DataFrame(rule).sort_values("next12_vs_last12_ratio", ascending=False)

capacity_actions.to_csv(OUT/"capacity_actions_by_route.csv", index=False)

# 24) Capacity action counts
plt.figure(figsize=(6,4))
capacity_actions["action"].value_counts().plot(kind="bar", color=["#2ca02c","#ff7f0e","#1f77b4"])
plt.title("Capacity Actions (Next 12m)"); plt.tight_layout()
plt.savefig(VIZ/"24_capacity_actions_counts.png", dpi=180); plt.close()




In [ ]:
# ========= 11. PSO ELIGIBILITY SCORING =========
# Components: low demand, island/over-water proxy, long distance, weak recovery
# Recovery: post vs pre as earlier
def post_vs_pre_pct_route(g):
    pre = g[g["date"]<="2020-02-29"]["pax"].mean()
    post= g[g["date"]>="2021-01-31"]["pax"].mean()
    return (post - pre) / (pre+1e-9)

ru = panel_ud.merge(route_meta2, on="route_ud", how="left")
rec = ru.groupby("route_ud").apply(post_vs_pre_pct_route).reset_index(name="post_vs_pre_pct")
avg_p = panel_ud.groupby("route_ud")["pax"].mean().reset_index(name="avg_pax")
score_df = (route_meta2.merge(avg_p, on="route_ud", how="left")
                        .merge(rec, on="route_ud", how="left"))
score_df["low_demand"] = 1 - scale01(score_df["avg_pax"])
score_df["long_dist"] = scale01(score_df["distance_km"].fillna(score_df["distance_km"].median()))

score_df["weak_recovery"] = 1 - scale01(np.nan_to_num(score_df["post_vs_pre_pct"], nan=0.0))
score_df["is_island"] = score_df["island_any"].astype(int)

# Weights (tunable)
score_df["pso_score"] = (0.35*score_df["low_demand"] +
                         0.30*score_df["is_island"] +
                         0.20*score_df["long_dist"] +
                         0.15*score_df["weak_recovery"])

score_df = score_df.sort_values("pso_score", ascending=False)
score_df.to_csv(OUT/"pso_candidate_routes_ranked.csv", index=False)

# PSO top-25 bar
plt.figure(figsize=(8,6))
sns.barplot(data=score_df.head(25), x="pso_score", y="route_ud", palette="Blues_r")

plt.title("PSO Candidate Score – Top 25")
plt.tight_layout(); plt.savefig(VIZ/"25_pso_top25.png", dpi=180); plt.close()




In [ ]:
# ========= 12. RESIDUAL DIAGNOSTICS FOR SAMPLE ROUTE =========
if len(results_df):
    rid = results_df["route_ud"].iloc[0]
    df_r = mdf[mdf["route_ud"]==rid].sort_values("date").dropna(subset=["pax_lag12"])
    tr, te = df_r.iloc[:-HOLDOUT], df_r.iloc[-HOLDOUT:]
    # fit sarimax again
    sar = SARIMAX(tr["pax"], order=(1,1,1), seasonal_order=(1,1,1,12), enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    pred = sar.get_forecast(steps=HOLDOUT).predicted_mean
    resid = te["pax"].values - pred.values
    plt.figure(figsize=(10,3)); plt.plot(te["date"], resid)
    plt.title(f"Residuals (SARIMAX) – {rid}"); plt.tight_layout()
    plt.savefig(VIZ/"26_residuals_example.png", dpi=180); plt.close()


In [ ]:
# ========= 13. SUMMARY EXPORTS =========
nat.to_csv(OUT/"national_monthly_totals.csv", index=False)
route_meta.to_csv(OUT/"route_meta_distance.csv", index=False)


In [ ]:
# Write a README-like summary
with open(OUT/"SUMMARY.txt","w") as f:
    f.write(
        "Outputs generated:\n"
        "- visuals/: 26+ PNGs covering temporal, seasonal, spatial, model & policy views\n"
        "- model_accuracy_comparison_by_route.csv: Baseline vs SARIMAX vs XGB/GBR vs LSTM (holdout metrics)\n"
        "- future_12m_forecasts_by_route.csv: Deployed 12-month forecasts per route\n"
        "- lf_model_metrics.csv: Load-factor model accuracy per route\n"
        "- capacity_actions_by_route.csv: Increase/Maintain/Reduce guidance\n"
        "- pso_candidate_routes_ranked.csv: PSO score & ranking\n"
        "- national_monthly_totals.csv, route_meta_distance.csv: supporting tables\n"
    )

print("All done. See the 'outputs/' folder (visuals, metrics, forecasts, PSO ranking).")


All done. See the 'outputs/' folder (visuals, metrics, forecasts, PSO ranking).


In [ ]:
# ========= 14. SAVE-ALL EXPORTS (tables, visuals, archives, slides, report, manifest) =========
import io, hashlib, zipfile
from datetime import datetime
from pathlib import Path
from contextlib import suppress

from pandas import ExcelWriter

try:
    from pptx import Presentation
    from pptx.util import Inches, Pt
    from pptx.enum.text import PP_ALIGN
except Exception:
    Presentation = None

try:
    from docx import Document
    from docx.shared import Inches as DocxInches, Pt as DocxPt
except Exception:
    Document = None

EXPORTS = OUT / "exports"
EXPORTS.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
excel_path = EXPORTS / f"UK_Domestic_Route_Analysis_Summary_{timestamp}.xlsx"
pptx_path  = EXPORTS / f"UK_Domestic_Route_Analysis_Slides_{timestamp}.pptx"
docx_path  = EXPORTS / f"UK_Domestic_Route_Analysis_Report_{timestamp}.docx"
zip_path   = EXPORTS / f"UK_Domestic_Route_Visuals_{timestamp}.zip"

zip_all    = EXPORTS / f"UK_Domestic_Route_Full_Export_{timestamp}.zip"
manifest   = EXPORTS / f"manifest_{timestamp}.json"

# ---------- Helper: safe writer ----------
def _add_sheet(writer, df, name):
    if df is None or not isinstance(df, pd.DataFrame) or df.empty:
        return
    # Excel sheetname max 31 chars
    name = (name[:28] + "...") if len(name) > 31 else name
    df.to_excel(writer, sheet_name=name, index=False)

# ---------- 14.1 Write all key DataFrames to a single multi-sheet Excel ----------
with ExcelWriter(excel_path, engine="openpyxl", mode="w") as xw:
    # Model metrics (if present)


    _add_sheet(xw, locals().get("results_df"),        "Model_Accuracy")
    _add_sheet(xw, locals().get("lf_metrics_df"),     "LF_Model_Accuracy")
    # Forecasts (deployed next 12 months)
    _add_sheet(xw, locals().get("future_fc"),         "Forecasts_Next12M")
    # Capacity actions + PSO
    _add_sheet(xw, locals().get("capacity_actions"),  "Capacity_Actions")
    _add_sheet(xw, locals().get("score_df"),          "PSO_Ranking")
    # Core supporting tables
    _add_sheet(xw, locals().get("nat"),               "National_Monthly")
    _add_sheet(xw, locals().get("route_meta"),        "Route_Distance_Meta")
    # Seasonality table if you created it
    _add_sheet(xw, locals().get("season_idx").reset_index().rename(columns={"index":"month"}) if "season_idx" in locals() else None, "Seasonality_Index")
    # Structural shifts (if available)

    _add_sheet(xw, locals().get("shifts"),            "Structural_Shifts")

# Also save as individual CSVs for diffs / Git
def _save_csv(df, name):
    if df is None or not isinstance(df, pd.DataFrame) or df.empty:
        return
    df.to_csv(OUT / f"{name}.csv", index=False)

_save_csv(locals().get("results_df"),       "model_accuracy_comparison_by_route")  # already saved earlier; safe to overwrite
_save_csv(locals().get("lf_metrics_df"),    "lf_model_metrics")
_save_csv(locals().get("future_fc"),        "future_12m_forecasts_by_route")
_save_csv(locals().get("capacity_actions"), "capacity_actions_by_route")
_save_csv(locals().get("score_df"),         "pso_candidate_routes_ranked")

_save_csv(locals().get("nat"),              "national_monthly_totals")
_save_csv(locals().get("route_meta"),       "route_meta_distance")
if "season_idx" in locals():
    season_idx.reset_index().rename(columns={"index":"month", 0:"index"}).to_csv(OUT / "national_seasonality_index.csv", index=False)
if "shifts" in locals():
    shifts.to_csv(OUT / "route_structural_shifts.csv", index=False)

# ---------- 14.2 Zip ALL visuals in outputs/visuals ----------
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(Path(VIZ).glob("*.png")):
        zf.write(p, arcname=p.name)
    for p in sorted(Path(VIZ).glob("*.svg")):


        zf.write(p, arcname=p.name)
    for p in sorted(Path(VIZ).glob("*.jpg")):
        zf.write(p, arcname=p.name)

# ---------- 14.3 Build a PowerPoint deck with all visuals ----------
if Presentation is not None:
    prs = Presentation()
    # a simple title slide
    title_slide_layout = prs.slide_layouts[0]
    sld = prs.slides.add_slide(title_slide_layout)
    sld.shapes.title.text = "UK Domestic Air Route Analysis"
    sld.placeholders[1].text = f"Automated summary generated on {datetime.now():%d %b %Y %H:%M}\nFiles in: {OUT.resolve()}"

    # one slide per visual
    blank = prs.slide_layouts[6]  # blank layout
    for img in sorted(Path(VIZ).glob("*.png")):
        s = prs.slides.add_slide(blank)
        # Title textbox
        tx = s.shapes.add_textbox(Inches(0.5), Inches(0.2), Inches(9), Inches(0.6))
        tf = tx.text_frame
        tf.clear()
        p = tf.paragraphs[0]
        p.text = img.name
        p.font.size = Pt(16)
        p.font.bold = True
        p.alignment = PP_ALIGN.LEFT
        # Picture

        s.shapes.add_picture(str(img), Inches(0.5), Inches(1.0), width=Inches(9.0))
    prs.save(pptx_path)

# ---------- 14.4 Create a short Word report with links & key stats ----------
def _fmt(v):
    try:
        return f"{v:,.2f}"
    except Exception:
        return str(v)

if Document is not None:
    doc = Document()
    doc.add_heading("UK Domestic Air Route Analysis – Summary", level=0)
    doc.add_paragraph(f"Generated: {datetime.now():%d %b %Y %H:%M}")
    doc.add_paragraph(f"Output folder: {OUT.resolve()}")

    # Key metrics blocks (safe if dataframes present)


    if "results_df" in locals() and isinstance(results_df, pd.DataFrame) and not results_df.empty:
        best_overall = results_df.sort_values("sar_RMSE").head(10)
        doc.add_heading("Model Accuracy (Top 10 by SARIMAX RMSE)", level=1)
        t = doc.add_table(rows=1, cols=4)
        hdr = t.rows[0].cells
        hdr[0].text = "Route"
        hdr[1].text = "Best Model"
        hdr[2].text = "Best RMSE (approx)"
        hdr[3].text = "Best MAPE%"
        for _, r in best_overall.iterrows():
            row = t.add_row().cells
            row[0].text = str(r["route_ud"])
            row[1].text = str(r["best_model"])
            # pick the best metric among candidates
            rmse_candidates = [r.get("sn_RMSE"), r.get("sar_RMSE"), r.get("xg_RMSE"), r.get("lstm_RMSE")]


            mape_candidates = [r.get("sn_MAPE%"), r.get("sar_MAPE%"), r.get("xg_MAPE%"), r.get("lstm_MAPE%")]
            row[2].text = _fmt(min([m for m in rmse_candidates if pd.notna(m)]))
            row[3].text = _fmt(min([m for m in mape_candidates if pd.notna(m)]))

    if "capacity_actions" in locals() and isinstance(capacity_actions, pd.DataFrame) and not capacity_actions.empty:
        doc.add_heading("Capacity Actions (Counts)", level=1)
        counts = capacity_actions["action"].value_counts()
        for lbl, cnt in counts.items():
            doc.add_paragraph(f"{lbl}: {cnt}", style="List Bullet")

    if "score_df" in locals() and isinstance(score_df, pd.DataFrame) and not score_df.empty:
        doc.add_heading("Top PSO Candidates (Top 10 by Score)", level=1)
        t = doc.add_table(rows=1, cols=2)

        t.rows[0].cells[0].text = "Route"
        t.rows[0].cells[1].text = "PSO Score"
        for _, r in score_df.head(10).iterrows():
            row = t.add_row().cells
            row[0].text = str(r["route_ud"])
            row[1].text = _fmt(r["pso_score"])

    doc.add_heading("Files", level=1)
    doc.add_paragraph(f"Excel summary: {excel_path.name}")
    doc.add_paragraph(f"PowerPoint:    {pptx_path.name if Presentation is not None else '(pptx not created – python-pptx missing)'}")
    doc.add_paragraph(f"Visuals ZIP:   {zip_path.name}")
    doc.add_paragraph(f"Full ZIP:      {zip_all.name}")

    doc.save(docx_path)










In [ ]:
# ---------- 14.5 Create a FULL ZIP export (tables + visuals + models) ----------
def _add_file(zf, path: Path, arcbase=""):
    if path.exists() and path.is_file():
        zf.write(path, arcname=str(Path(arcbase) / path.name))

with zipfile.ZipFile(zip_all, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    # Excel, Word, PPTX
    _add_file(zf, excel_path, "summary")
    if Presentation is not None: _add_file(zf, pptx_path, "summary")
    if Document is not None:    _add_file(zf, docx_path, "summary")
    # All CSVs (top-level OUT/*.csv)
    for f in OUT.glob("*.csv"):
        _add_file(zf, f, "csv")
    # Visuals bundle zip

    _add_file(zf, zip_path, "visuals")
    # Raw visuals as well
    for img in VIZ.glob("*.png"):
        _add_file(zf, img, "visuals/png")
    for img in VIZ.glob("*.svg"):
        _add_file(zf, img, "visuals/svg")
    # Models/debug if any
    for f in MODELS.glob("*"):
        if f.is_file():
            _add_file(zf, f, "models")
    dbg = OUT / "debug"
    if dbg.exists():
        for f in dbg.glob("*"):
            if f.is_file():
                _add_file(zf, f, "debug")



In [ ]:
# ---------- 14.6 Write a manifest with file hashes ----------
def md5sum(p: Path) -> str:
    h = hashlib.md5()
    with open(p, "rb") as fh:
        for chunk in iter(lambda: fh.read(65536), b""): h.update(chunk)
    return h.hexdigest()

manifest_data = {
    "generated_at": datetime.now().isoformat(),
    "root": str(OUT.resolve()),
    "artifacts": []
}
for p in sorted(list(OUT.glob("**/*"))):
    if p.is_file():
        rel = str(p.relative_to(OUT))
        with suppress(Exception):
            manifest_data["artifacts"].append({
                "path": rel,


                "bytes": p.stat().st_size,
                "md5": md5sum(p)
            })

import json
with open(manifest, "w") as f:
    json.dump(manifest_data, f, indent=2)

print(f"\nExport complete:\n  Excel:   {excel_path}\n  PPTX:    {pptx_path if Presentation is not None else '(pptx skipped)'}\n  DOCX:    {docx_path if Document is not None else '(docx skipped)'}\n  VIS ZIP: {zip_path}\n  FULL ZIP:{zip_all}\n  Manifest:{manifest}\n")





Export complete:
  Excel:   outputs/exports/UK_Domestic_Route_Analysis_Summary_20251112_1229.xlsx
  PPTX:    (pptx skipped)
  DOCX:    (docx skipped)
  VIS ZIP: outputs/exports/UK_Domestic_Route_Visuals_20251112_1229.zip
  FULL ZIP:outputs/exports/UK_Domestic_Route_Full_Export_20251112_1229.zip
  Manifest:outputs/exports/manifest_20251112_1229.json



In [ ]:
!pip install python-docx
import os
import re
import json
import math
import glob
import textwrap
from datetime import datetime
from pathlib import Path

import pandas as pd

from docx import Document
from docx.shared import Inches, Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.section import WD_SECTION
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.5 MB/s eta 0:00:00


In [ ]:
def _exists_df(path):
    try:
        return Path(path).is_file() and (pd.read_csv(path).shape[0] >= 0 or True)
    except Exception:
        return False


In [ ]:
def _read_csv_safe(path):
    try:
        return pd.read_csv(path)
    except Exception:
        return None


In [ ]:

def _add_heading(doc, text, level=1):
    h = doc.add_heading(text, level=level)
    return h


In [ ]:

def _add_para(doc, text, italic=False, bold=False, align=None, style=None):
    p = doc.add_paragraph(style=style) if style else doc.add_paragraph()
    run = p.add_run(text)
    run.italic = italic
    run.bold = bold
    if align:
        p.alignment = align
    return p


In [ ]:

def _add_bullets(doc, items, level=0):
    for it in items:
        p = doc.add_paragraph(it, style="List Bullet")
        if level:
            p.style = f"List Bullet {level}"


In [ ]:

def _add_numbered(doc, items):
    for it in items:
        doc.add_paragraph(it, style="List Number")


In [ ]:
def _add_table_from_df(doc, df, title=None, col_width_in=None, max_rows=40, align=WD_TABLE_ALIGNMENT.CENTER):
    if df is None or df.empty:
        return None
    df = df.copy()
    if max_rows and len(df) > max_rows:
        df = df.head(max_rows)
    if title:
        _add_para(doc, title, bold=True)
    rows, cols = df.shape
    table = doc.add_table(rows=rows+1, cols=cols)
    table.alignment = align
    table.style = "Light Shading Accent 1"
    # header
    for j, c in enumerate(df.columns):
        table.cell(0, j).text = str(c)
    # rows
    for i in range(rows):
        for j in range(cols):
            val = df.iloc[i, j]
            table.cell(i+1, j).text = "" if pd.isna(val) else str(val)
    # widths
    if col_width_in:


        for j in range(cols):
            for i in range(rows+1):
                table.cell(i, j).width = Inches(col_width_in if isinstance(col_width_in, (int,float)) else col_width_in[j])
    return table




In [ ]:
def _insert_picture(doc, image_path, caption=None, width_in=6.5, align=WD_ALIGN_PARAGRAPH.CENTER):
    if not Path(image_path).is_file():
        return None
    p = doc.add_paragraph()
    r = p.add_run()
    r.add_picture(str(image_path), width=Inches(width_in))
    p.alignment = align
    if caption:
        cap = doc.add_paragraph(caption)
        cap.alignment = align
        cap.runs[0].italic = True
    return p



In [ ]:
def _add_toc(doc):
    """
    Adds a Word Table of Contents field.
    Note: The ToC will populate after opening the document in Word and pressing F9 / "Update Table".
    """
    p = doc.add_paragraph()
    run = p.add_run()
    fld = OxmlElement('w:fldSimple')
    fld.set(qn('w:instr'), 'TOC \\o "1-3" \\h \\z \\u')
    run._r.append(fld)
    return p


In [ ]:

def _fmt_pct(x, digits=1):
    try:
        return f"{float(x)*100:.{digits}f}%"
    except Exception:
        return str(x)


In [ ]:

def _fmt_num(x, digits=0):
    try:
        return f"{float(x):,.{digits}f}"
    except Exception:
        return str(x)


In [ ]:
def _value_counts_df(series, name_value="value", name_count="count"):
    if series is None:
        return pd.DataFrame()
    vc = series.value_counts(dropna=False)
    return pd.DataFrame({name_value: vc.index.astype(str), name_count: vc.values})


In [ ]:
def build_word_report(outputs_dir="outputs", report_name=None):
    """
    Build a comprehensive Word report using CSVs & images in outputs/ and outputs/visuals/.
    """
    OUT = Path(outputs_dir)
    VIZ = OUT / "visuals"
    EXPORTS = OUT / "exports"
    EXPORTS.mkdir(parents=True, exist_ok=True)

    if report_name is None:
        report_name = f"UK_Domestic_Route_Analysis_Report_{datetime.now():%Y%m%d_%H%M}.docx"

    report_path = EXPORTS / report_name

    df_nat     = _read_csv_safe(OUT / "national_monthly_totals.csv")
    df_acc     = _read_csv_safe(OUT / "model_accuracy_comparison_by_route.csv")
    df_lfmet   = _read_csv_safe(OUT / "lf_model_metrics.csv")
    df_fc      = _read_csv_safe(OUT / "future_12m_forecasts_by_route.csv")
    df_caps    = _read_csv_safe(OUT / "capacity_actions_by_route.csv")
    df_pso     = _read_csv_safe(OUT / "pso_candidate_routes_ranked.csv")
    df_rmeta   = _read_csv_safe(OUT / "route_meta_distance.csv")
    df_season  = _read_csv_safe(OUT / "national_seasonality_index.csv")
    df_shifts  = _read_csv_safe(OUT / "route_structural_shifts.csv")

    doc = Document()

    # Title page
    title = doc.add_paragraph()
    run = title.add_run("Machine Learning‑Based Load Factor and Passenger Demand Analysis\nfor UK Domestic Air Route Optimisation")
    run.font.size = Pt(20); run.bold = True
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER

    _add_para(doc, f"Generated: {datetime.now():%d %b %Y, %H:%M}", align=WD_ALIGN_PARAGRAPH.CENTER)
    _add_para(doc, f"Output directory: {OUT.resolve()}", align=WD_ALIGN_PARAGRAPH.CENTER)


    doc.add_page_break()

    # Table of contents
    _add_heading(doc, "Table of Contents", level=1)
    _add_toc(doc)
    doc.add_page_break()

    # Executive Summary
    _add_heading(doc, "1. Executive Summary", level=1)
    bullets = []
    if df_acc is not None and not df_acc.empty:
        n_routes = df_acc["route_ud"].nunique()
        best_counts = df_acc["best_model"].value_counts()


        top_model = best_counts.index[0] if len(best_counts) else "N/A"
        bullets.append(f"Model evaluation covered **{n_routes}** routes; the most frequently best-performing model: **{top_model}**.")
    if df_caps is not None and not df_caps.empty:
        action_counts = df_caps["action"].value_counts()
        bullets.append("Capacity actions suggested (next 12 months): " +
                       ", ".join([f"**{k}**: {v}" for k,v in action_counts.items()]))
    if df_pso is not None and not df_pso.empty:
        top_pso = df_pso.head(5)["route_ud"].tolist()
        bullets.append("Top PSO candidates (by score): " + "; ".join(top_pso))
    if df_nat is not None and not df_nat.empty:
        bullets.append(f"Total national observations: **{len(df_nat)} monthly points**.")
    if bullets:
        _add_bullets(doc, bullets)



    # Methods & Data
    _add_heading(doc, "2. Data, Methods, and Assumptions", level=1)
    _add_bullets(doc, [
        "Data sources: Monthly domestic passenger flows (2015–2025), UK airports registry (ICAO/IATA & coordinates), 2019 Q1 load‑factor (LF) distributions by reporting airport.",
        "Cleaning: Name normalisation; undirected route keys; monthly panel; airport geocoding and great‑circle distances.",
        "Features: Lag features (1,3,6,12), cyclic seasonality (sin/cos), disruption windows (COVID shock/recovery), route distance, island/over‑water hints.",
        "Models: Baselines (Seasonal‑Naïve) vs SARIMAX (ARIMA seasonal), XGBoost/GBR (tabular ML), and LSTM (selected routes).",


        "LF estimation: Weighted average from LF bucket shares (2019 Q1) with time‑adjustment by airport throughput vs 2019 baseline; optional ML prediction for LF.",
        "PSO scoring: Composite of low‑demand, island/over‑water, long distance, and weak post‑COVID recovery (weights configurable)."
    ])

    # EDA
    _add_heading(doc, "3. Exploratory Data Analysis (Temporal, Spatial, Seasonal)", level=1)

    # Key figures (insert if they exist)
    key_figs = [

        ("01_national_monthly.png", "National monthly domestic demand"),
        ("03_seasonality_index.png", "Seasonality index (Avg = 1.0)"),
        ("06_route_flow_heatmap.png", "Route flow heatmap (log pax) – top airports"),
        ("05_top15_routes_ts.png", "Top 15 routes – monthly passengers"),
        ("08_dist_vs_avgpax.png", "Distance vs average monthly passengers"),
        ("07_national_monthly_heatmap.png", "Monthly demand heatmap (national)"),
        ("13_post_vs_pre_losses.png", "Largest negative post‑vs‑pre changes (%)"),
        ("14_post_vs_pre_gains.png", "Largest positive post‑vs‑pre changes (%)")
    ]
    for fname, caption in key_figs:
        _insert_picture(doc, VIZ/fname, caption=caption)

    # If available, add a short stats paragraph

    if df_nat is not None and not df_nat.empty:
        df_nat["date"] = pd.to_datetime(df_nat["date"])
        tot = df_nat["pax"].sum()
        avg_m = df_nat["pax"].mean()
        peak = df_nat.loc[df_nat["pax"].idxmax()]
        trough = df_nat.loc[df_nat["pax"].idxmin()]
        _add_para(doc,
            f"National sample: total passengers across series ≈ **{_fmt_num(tot)}**, "
            f"monthly average ≈ **{_fmt_num(avg_m)}**. "
            f"Peak {peak['date']:%b %Y}: **{_fmt_num(peak['pax'])}**; "
            f"Trough {trough['date']:%b %Y}: **{_fmt_num(trough['pax'])}**."
        )

    # Model Evaluation
    _add_heading(doc, "4. Forecasting Models & Accuracy", level=1)
    if df_acc is not None and not df_acc.empty:
        # Best model distribution
        _add_para(doc, "Best model frequency across routes:", bold=True)
        best_dist = _value_counts_df(df_acc["best_model"], "model", "count")
        _add_table_from_df(doc, best_dist, title=None)

        # Top‑10 by SARIMAX RMSE (lower is better)


        top10 = df_acc.sort_values("sar_RMSE").head(10)[["route_ud","best_model","sar_RMSE","sar_MAPE%","xg_RMSE","lstm_RMSE"]]
        _add_table_from_df(doc, top10, title="Top‑10 routes by SARIMAX RMSE")

        # Feature importance figure if present
        _insert_picture(doc, VIZ/"21_feature_importance.png", caption="Feature importance (example route)")

    # Demand Forecasts & Capacity Actions
    _add_heading(doc, "5. Demand Forecasts & Resource Allocation", level=1)
    if df_fc is not None and not df_fc.empty:
        # Summarise next‑12m demand vs baseline by number of routes
        n_routes = df_fc["route_ud"].nunique()
        _add_para(doc, f"Generated 12‑month forecasts for **{n_routes}** routes.",
bold=False)
    if df_caps is not None and not df_caps.empty:
        counts = df_caps["action"].value_counts().reset_index()
        counts.columns = ["Action","Count"]
        _add_table_from_df(doc, counts, title="Capacity Actions – Counts (Next 12 Months)")
        _insert_picture(doc, VIZ/"24_capacity_actions_counts.png", caption="Capacity action counts (chart)")

        # Show top routes to increase and to reduce
        inc = df_caps[df_caps["action"].str.contains("Increase", na=False)].head(15)
        red = df_caps[df_caps["action"].str.contains("Reduce", na=False)].head(15)

        if not inc.empty:
            _add_table_from_df(doc, inc[["route_ud","next12_vs_last12_ratio","lf_median_pred","action"]],
                               title="Top routes flagged to Increase / Upgauge (sample)")
        if not red.empty:
            _add_table_from_df(doc, red[["route_ud","next12_vs_last12_ratio","lf_median_pred","action"]],
                               title="Top routes flagged to Reduce / Downgauge (sample)")

# Load Factor Findings
    _add_heading(doc, "6. Load Factor (LF) Estimation & Prediction", level=1)
    _insert_picture(doc, VIZ/"22_lf_proxy_distribution.png", caption="LF proxy distribution across route‑months")
    if df_lfmet is not None and not df_lfmet.empty:
        _add_para(doc, "LF model accuracy (per‑route, holdout):", bold=True)
        lf_stats = df_lfmet[["MAE","RMSE","MAPE%"]].median().to_frame("Median").reset_index().rename(columns={"index":"Metric"})
        _add_table_from_df(doc, lf_stats, title=None)
        # Show a few best/worst LF routes by RMSE
        best_lf = df_lfmet.sort_values("RMSE").head(10)
        worst_lf= df_lfmet.sort_values("RMSE").tail(10)


        _add_table_from_df(doc, best_lf, title="Best LF prediction routes (lowest RMSE) – sample")
        _add_table_from_df(doc, worst_lf, title="Challenging LF prediction routes (highest RMSE) – sample")

    # PSO Eligibility
    _add_heading(doc, "7. PSO Eligibility – Critical Routes & Policy Insights", level=1)
    _insert_picture(doc, VIZ/"25_pso_top25.png", caption="PSO candidate score – Top 25")
    if df_pso is not None and not df_pso.empty:
        _add_table_from_df(doc, df_pso.head(25), title="PSO Candidate Ranking (Top 25)")

    # Structural Shifts
    _add_heading(doc, "8. Structural Shifts & Disruptions (e.g., COVID‑19)", level=1)


    if df_shifts is not None and not df_shifts.empty:
        # show top anomalies
        anom = df_shifts[df_shifts["cusum_anomaly"]==True].copy()
        anom = anom.sort_values("post_vs_pre_pct").head(20)
        if not anom.empty:
            _add_table_from_df(doc, anom, title="Routes with strong structural anomalies (sample)")
    # Add optional figure if you exported residuals/other diagnostics
    if (VIZ/"26_residuals_example.png").is_file():
        _insert_picture(doc, VIZ/"26_residuals_example.png", caption="Residual diagnostics (example route)")

    # Appendices


    _add_heading(doc, "Appendix A – Data Tables", level=1)
    if df_nat is not None and not df_nat.empty:
        _add_table_from_df(doc, df_nat.tail(24), title="National monthly totals – last 24 rows")
    if df_rmeta is not None and not df_rmeta.empty:
        _add_table_from_df(doc, df_rmeta.head(30), title="Route distance meta – sample")

    _add_heading(doc, "Appendix B – Figures in this Report", level=1)
    # list all visuals under outputs/visuals
    all_imgs = sorted(list((VIZ).glob("*.png")))
    if all_imgs:
        listing = "\n".join([f"• {p.name}" for p in all_imgs])
        _add_para(doc, listing)



    # Methods/Assumptions (detailed)
    _add_heading(doc, "Appendix C – Detailed Methods & Assumptions", level=1)
    details = """
    Demand modelling:
      • Baselines (Seasonal‑Naïve) compare last season’s values.
      • SARIMAX: small grid around (p,d,q) × (P,D,Q,12) with non‑enforced stationarity for robustness.
      • XGBoost/GBR: lagged demand (1,3,6,12), seasonal sin/cos, distance, COVID flags as regressors.
      • LSTM: windowed univariate networks on selected top‑traffic routes (runtime‑bounded).
    Load factors:


      • Estimated from 2019 Q1 bucket shares using mid‑points, time‑scaled by airport throughput vs 2019.
      • LF ML predicts LF as a function of forecast demand, distance, month (sin/cos), COVID flags.
    PSO scoring:
      • Composite of (low demand, island/over‑water proxy, long distance, weak recovery).
      • Weights tunable; outputs meant to be evidence‑guidance, not prescriptive.
    """
    _add_para(doc, textwrap.dedent(details))

    # Save document
    doc.save(report_path)
    print(f"Word report saved to: {report_path.resolve()}")
    return report_path

In [ ]:
report_path = build_word_report()
print(f"Report generated at: {report_path}")

Word report saved to: /content/outputs/exports/UK_Domestic_Route_Analysis_Report_20251112_1601.docx
Report generated at: outputs/exports/UK_Domestic_Route_Analysis_Report_20251112_1601.docx


In [ ]:
pip install pandas numpy statsmodels scikit-learn xgboost openpyx

ERROR: Could not find a version that satisfies the requirement openpyx (from versions: none)
ERROR: No matching distribution found for openpyx
